# CoachBot AI — exploration notebook

Walks through the pipeline used by the Streamlit app: profile → safety screen → prompt → Gemini → validated plan → charts.

Runs **offline with the demo engine**; set `GEMINI_API_KEY` in your environment to call Gemini for real.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
from coachbot.features import FEATURES, FEATURES_BY_ID
from coachbot.gemini_client import CoachEngine, Params
from coachbot import safety, charts
from coachbot.prompts import build_prompt, system_instruction
print(len(FEATURES), 'features:', [f.id for f in FEATURES])

## 1. Athlete profile (the brief's example input)
*Cricket player, right-arm fast bowler, recovering from knee strain. Needs low-impact conditioning and flexibility improvement.*

In [ ]:
profile = dict(name='Ravi', age=15, gender='Male', sport='Cricket', position='Fast bowler', experience='Club / school team',
               injuries=['Knee strain'], injury_notes='right knee', intensity='Moderate', styles=['Bodyweight','Yoga & mobility'],
               equipment='Basic (ball, cones, bands)', diet='Vegetarian', calories=2400, allergies='', goal='Post-injury recovery',
               tone='Calm Mentor', language='English')

## 2. Safety screen → hard constraints

In [ ]:
screen = safety.screen_input(profile, '')
screen.allowed, screen.constraints

In [ ]:
safety.screen_input(profile, 'I hit my head yesterday and feel dizzy after the fall')

## 3. The exact prompt sent to Gemini

In [ ]:
feature = FEATURES_BY_ID['injury_recovery']
extras = {x.key: x.default for x in feature.extras}
print(system_instruction(profile))
print('-'*80)
print(build_prompt(feature, profile, extras, screen.constraints))

## 4. Generate (Gemini if a key is set, otherwise demo engine)

In [ ]:
engine = CoachEngine(os.environ.get('GEMINI_API_KEY'), 'gemini-flash-latest')
result = engine.generate(feature, profile, extras, Params.for_feature(feature))
print(result.status, f'{result.latency_s:.2f}s')
print(json.dumps(result.plan.model_dump(), indent=2)[:2000])

## 5. Temperature experiment
Same prompt at 0.2 / 0.5 / 0.9 — compare length and lexical variety.

In [ ]:
import re
for t in (0.2, 0.5, 0.9):
    r = engine.generate(FEATURES_BY_ID['mental_focus'], profile, {x.key: x.default for x in FEATURES_BY_ID['mental_focus'].extras}, Params(t, 0.95, 50))
    words = re.findall(r"[a-z']+", json.dumps(r.plan.model_dump()).lower())
    print(f'T={t}: {len(words)} words, unique ratio {len(set(words))/len(words):.2f} | {r.plan.headline}')

## 6. Visualise

In [ ]:
charts.load_landscape(result.plan).show()
charts.weekly_load(result.plan).show()